# 02 — Original 12-condition benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokhanturan/apsp-knn-benchmark/blob/main/notebooks/02_original_benchmark.ipynb)

This notebook reruns the 12 original graph conditions. It uses the publication protocol: 3 warm-ups, 30 timing observations, randomized method order, calibrated batching for short calls, and 30 fresh-process memory measurements.

**Runtime note:** full reproduction is intentionally computationally non-trivial. Absolute times and RSS values will vary across Colab sessions/hardware.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_NAME = "apsp-knn-benchmark"
REPO_URL = "https://github.com/gokhanturan/apsp-knn-benchmark.git"

# Colab: clone the repository if the notebook was opened directly from GitHub.
if Path('/content').exists() and not (Path('/content') / REPO_NAME).exists():
    subprocess.run(['git', 'clone', REPO_URL, str(Path('/content') / REPO_NAME)], check=True)

if (Path('/content') / REPO_NAME).exists():
    ROOT = Path('/content') / REPO_NAME
else:
    # Local/Jupyter execution from repo/notebooks or repo root.
    cwd = Path.cwd().resolve()
    ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

os.chdir(ROOT)
print('Repository root:', ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import sys, subprocess
from pathlib import Path
import pandas as pd
import numpy as np

GRAPHS = ROOT / 'graphs'
OUT = ROOT / 'reproduced_results'
OUT.mkdir(exist_ok=True)

DATASETS = ['wine', 'diabetes', 'breast_cancer', 'digits']
K_VALUES = [5, 10, 20]
ALGORITHMS = ['floyd_warshall', 'johnson', 'dijkstra']


## Runtime + numerical-equivalence measurements

In [ ]:
runtime_files = []
correctness_files = []
for dataset in DATASETS:
    for k in K_VALUES:
        graph = GRAPHS / f'{dataset}_k{k}.npz'
        rt_out = OUT / f'rt_{dataset}_k{k}.csv'
        corr_out = OUT / f'corr_{dataset}_k{k}.csv'
        cmd = [
            sys.executable, 'src/runtime_condition.py',
            '--graph', str(graph), '--dataset', dataset, '--k', str(k),
            '--repeats', '30', '--warmups', '3', '--random-seed', '20260804',
            '--output', str(rt_out), '--correctness-output', str(corr_out)
        ]
        subprocess.run(cmd, check=True)
        runtime_files.append(rt_out)
        correctness_files.append(corr_out)

runtime_raw = pd.concat([pd.read_csv(p) for p in runtime_files], ignore_index=True)
correctness = pd.concat([pd.read_csv(p) for p in correctness_files], ignore_index=True)
runtime_raw.to_csv(OUT / 'runtime_raw_30.csv', index=False)
correctness.to_csv(OUT / 'correctness_float64.csv', index=False)
print('Runtime rows:', len(runtime_raw))
print('Correctness checks passed:', int(correctness.passed.sum()), '/', len(correctness))

## Fresh-process memory measurements

In [ ]:
RUN_MEMORY = True
memory_files = []
if RUN_MEMORY:
    for dataset in DATASETS:
        for k in K_VALUES:
            graph = GRAPHS / f'{dataset}_k{k}.npz'
            for algorithm in ALGORITHMS:
                out = OUT / f'mem_{dataset}_k{k}_{algorithm}.csv'
                cmd = [
                    sys.executable, 'src/memory_condition.py',
                    '--graph', str(graph), '--dataset', dataset, '--k', str(k),
                    '--algorithm', algorithm, '--repeats', '30', '--output', str(out)
                ]
                subprocess.run(cmd, check=True)
                memory_files.append(out)
    memory_raw = pd.concat([pd.read_csv(p) for p in memory_files], ignore_index=True)
    memory_raw.to_csv(OUT / 'memory_raw_30.csv', index=False)
    print('Memory rows:', len(memory_raw))
else:
    print('Memory run skipped. Set RUN_MEMORY=True for the publication protocol.')

## Runtime summary

In [ ]:
def summarize_runtime(df):
    g = df.groupby(['dataset','k','algorithm'], as_index=False)
    out = g.agg(
        n_runs=('wall_time_s','size'),
        wall_median_s=('wall_time_s','median'),
        wall_mean_s=('wall_time_s','mean'),
        wall_sd_s=('wall_time_s','std'),
        cpu_median_s=('cpu_time_s','median'),
        cpu_wall_ratio_median=('cpu_wall_ratio','median'),
        nodes=('nodes','first'), edges=('edges','first')
    )
    return out

runtime_summary = summarize_runtime(runtime_raw)
display(runtime_summary.head(12))

pivot = runtime_summary.pivot_table(index=['dataset','k'], columns='algorithm', values='wall_median_s')
pivot['fastest'] = pivot.idxmin(axis=1)
pivot['dijkstra_speedup_vs_floyd'] = pivot['floyd_warshall'] / pivot['dijkstra']
display(pivot)
print('Fastest counts:', pivot.fastest.value_counts().to_dict())
print('Median Dijkstra speedup:', float(pivot.dijkstra_speedup_vs_floyd.median()))
print('Maximum Dijkstra speedup:', float(pivot.dijkstra_speedup_vs_floyd.max()))

A fresh Colab session is not expected to reproduce the manuscript’s absolute seconds exactly. The primary reproducibility targets are the experimental protocol, numerical equivalence, and structural/ordinal performance patterns.